# SpaceX Falcon 9 — Classification

Reproduces the model-comparison method used in the report from the IBM feature and target CSVs. Standardization is inside the training pipeline to avoid test-set leakage.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
X=pd.read_csv('dataset_part_3.csv')
y=pd.read_csv('dataset_part_2.csv')['Class']
x_train,x_test,y_train,y_test=train_test_split(X,y,test_size=.2,random_state=2)
print('Training:',len(x_train),'Testing:',len(x_test))

In [ ]:
settings={
'Logistic Regression':(LogisticRegression(max_iter=3000),{'clf__C':[.01,.1,1,10]}),
'SVM':(SVC(),{'clf__kernel':['linear','rbf','sigmoid'],'clf__C':[.1,1,10]}),
'Decision Tree':(DecisionTreeClassifier(random_state=2),{'clf__max_depth':[2,4,6,8],'clf__criterion':['gini','entropy']}),
'KNN':(KNeighborsClassifier(),{'clf__n_neighbors':[3,5,7,9]})
}
scores={}
predictions={}
for name,(model,params) in settings.items():
    search=GridSearchCV(Pipeline([('scale',StandardScaler()),('clf',model)]),params,cv=5,scoring='accuracy')
    search.fit(x_train,y_train)
    predicted=search.predict(x_test)
    scores[name]={'test_accuracy':accuracy_score(y_test,predicted),'cv_accuracy':search.best_score_,'parameters':search.best_params_}
    predictions[name]=predicted
display(pd.DataFrame(scores).T)

In [ ]:
best=max(scores,key=lambda n:scores[n]['test_accuracy'])
print('Best held-out model:',best)
print('Confusion matrix (actual rows 0,1 / predicted columns 0,1):')
print(confusion_matrix(y_test,predictions[best]))
print(classification_report(y_test,predictions[best]))